# A Quick PyTorch 2.0 Tutorial


## 30-second intro

PyTorch 2.0 is out!

With the main improvement being speed.

This comes via a single backwards-compatible line.

```python
torch.compile()
```

In other words, after you create your model, you can pass it to `torch.compile()` and in turn expect speedups in training and inference on newer GPUs (e.g. NVIDIA RTX 40 series, A100, H100, the newer the GPU the more noticeable the speedups).

> **Note:** There are plenty more upgrades within PyTorch 2.0 than just `torch.compile()` but since it's the main one, it's what we're going to focus on. For a full list of changes, see the [PyTorch 2.0 release notes](https://pytorch.org/blog/pytorch-2.0-release/).

### Will my old PyTorch code still work?

Yes, PyTorch 2.0 is backwards-compatible. The changes are mostly additive (new features).

That means if you already know PyTorch, such as via the [learnpytorch.io](https://learnpytorch.io) course, you can start using PyTorch 2.0 straight away. And your old PyTorch code will still work.

## Speedups

Ok so the focus of PyTorch 2.0 is speed, how much faster is it actually?

The PyTorch team ran tests across 163 open-source models from [Hugging Face Transformers](https://huggingface.co/docs/transformers/index), [timm](https://github.com/huggingface/pytorch-image-models) (PyTorch Image Models) and [TorchBench](https://github.com/pytorch/benchmark) (a curated set of popular code bases from across GitHub).

This is important because unless PyTorch 2.0 is faster on models people actually use, it’s not faster.

Using a mixture of AMP (automatic mixed precision or float16) training and float32 precision (higher precision requires more compute) the PyTorch team found that `torch.compile()` provides an average speedup of 43% in training on a NVIDIA A100 GPU.

Or 38% on timm, 76% on TorchBench and 52% on Hugging Face Transformers.

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/extras-pytorch-2-speedups.png" alt="speedups for PyTorch 2.0 across various model resources" width=650/>

*PyTorch 2.0 speedups across various models from different locations. *Source:* [PyTorch 2.0 announcement post](https://pytorch.org/get-started/pytorch-2.0/).*

## 3-minute overview

> **Note:** The following is adapted from [*A Quick Introduction to PyTorch 2.0*](https://www.mrdbourke.com/pytorch-2/) on mrdbourke.com, there's also an accompanying [video explainer on YouTube](https://youtu.be/WqLKfta5Ijw).

What's happening behind the scenes of `torch.compile()`?

`torch.compile()` is designed to "just work" but there are a few technologies behind it:
* TorchDynamo
* AOTAutograd
* PrimTorch
* TorchInductor

The [PyTorch 2.0 getting started notes](https://pytorch.org/get-started/pytorch-2.0/) explain these in more detail but from a high level the two main improvements `torch.compile()` offers are:
* Fusion (or operator fusion)
* Graph capture (or graph tracing)

### Fusion

Fusion, also known as **operator fusion** is one of the best ways to make deep learning models go brrrrrr (brrrrrr is the sound your GPUs fan make when your models are training).

Operator fusion condenses (like Dragon Ball Z) many operations into one (or many to less).

Why?

Modern GPUs have so much compute power they are often not compute limited, as in, the main bottleneck to training models is how fast you can get data from your CPU to your GPU.
This is known as bandwidth or memory bandwidth.

You want to reduce your bandwidth costs as much as possible.

And feed the data hungry GPUs with as much data as possible.

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/extras-memory-bandwidth-output-small.gif" alt="example of memory bandwidth costs transferring data on and off the GPU" width=950/>

So instead of performing an operation on a piece of data and then saving the result to memory (increased bandwidth costs), you chain together as many operations as possible via fusion.

A rough analogy would be using a blender to make a smoothie.

Most blenders are good at blending things (like GPUs are good at performing matrix multiplications).

Using a blender **without operator fusion** would be like adding each ingredient one by one and blending each time a new ingredient is added.
Not only is this insane, it increases your bandwidth cost.

The actual blending is fast each time (like GPU computations generally are) but you lose a bunch of time adding each ingredient one by one.

Using a blender **with operator fusion** is akin to using a blender by adding all the ingredients at the start (operator fusion) and then performing the blend once.

You lose a little time adding at the start but you gain all of the lost memory bandwidth time back.

### Graph capture

Graph capture I’m less confident explaining.

But the way I think about it is that graph capture or graph tracing is:

* Going through a series of operations that need to happen, such as the operations in a neural network.
* And capturing or tracing what needs to happen ahead of time.

Computing **without graph capture** is like going to a new area and following GPS directions turn by turn.

As a good human driver, you can follow the turns quite easily but you still have to think about each turn you take.

This is the equivalent to PyTorch having to look up what each operation does as it does it.

As in, to perform an addition, it has to look up what an addition does before it can perform it.

It does this quickly but there’s still non-zero overhead.

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/extras-graph-capture.gif" alt="Graph capture" width="950"/>

*Example of graph capture, mapping out the steps in a neural network and then capturing every operation that needs to happen ahead of time.*

Computing **with graph capture** is like driving through your own neighbourhood.

You barely think about what turns to make.

Sometimes you get out of the car and realise you can’t remember the last 5 minutes of the drive.

Your brain was functioning on autopilot, minimal overhead.

However, it took you some time upfront to remember how to drive to your house.

This is a caveat of graph capture, it takes a little time upfront to memorize the operations that need to happen but subsequent computations should be faster.

Of course, this is a quick high-level overview of what’s happening behind the scenes of torch.compile()but it's how I understand it.

For more on fusion and graph tracing, I’d recommend Horace He’s [*Making Deep Learning Go Brrrr From First Principles*](https://horace.io/brrr_intro.html) blog post.

## Things to note

Since PyTorch 2.0 was just released, there are a few limitations with some of the features.

One of the main ones being with exporting models.

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/extras-pytorch-2-limitations.png" alt="PyTorch 2 limitations" width=650/>

*There are a few caveats when using the PyTorch 2.0 features, such as not being about to export to mobile devices when using the `torch.compile()` default options. However, there are work arounds to this and improved exporting is on the PyTorch 2.x roadmap. *Source:* [PyTorch 2.0 announcement post](https://pytorch.org/get-started/pytorch-2.0/).*

However, these will likely be fixed in future releases.

Another main limitation is that because the features of PyTorch 2.0 are designed for newer hardware, old GPUs and desktop-class GPUs (e.g. NVIDIA RTX 30 series) will likely see less speedups than newer hardware.

## 0. Getting setup

To get setup we'll first check for PyTorch 2.x+ and install it if it's not available. 

You can see how to install PyTorch 2.x on your own system in the [PyTorch documentation](https://pytorch.org/get-started/locally/).

> **Note:** If you're running on Google Colab, you'll need to setup a GPU: runtime -> change runtime type -> hardware accelerator. The best speedups are on newer NVIDIA/AMD GPUs (this is because PyTorch 2.0 leverages newer GPU hardware) such as the NVIDIA A100 and above. This tutorial focuses on NVIDIA GPUs.

## 1. Get GPU info

Time to get GPU info.

Why?

Many of the speedups PyTorch 2.0 offers are best experienced on newer NVIDIA GPUs (we're focused on NVIDIA GPUs for now).

This is because PyTorch 2.0 takes advantage of the new hardware on newer GPUs.

How do you tell what's a newer GPU?

Generally, a *newer* GPU will have a compute capability score of 8.0 or higher.

You can see a list of [NVIDIA GPU compute capability scores](https://developer.nvidia.com/cuda-gpus) on NVIDIA's developer page.

Here are some scores of NVIDIA GPUs released in 2020 or later:

| **NVIDIA GPU** | **Compute capability score** | **GPU Type** | **Release year** | **Architecture** |
|----- |-----| -----| -----| -----| 
| RTX 4090 | 8.9 | Desktop-class | 2022 | [Ada Lovelace](https://www.nvidia.com/en-au/geforce/ada-lovelace-architecture/) |
| RTX 4080 | 8.9 | Desktop-class | 2022 | Ada Lovelace |
| RTX 4070 Ti | 8.9 | Desktop-class | 2022 | Ada Lovelace |
| RTX 3090 | 8.6 | Desktop-class | 2020 | [Ampere](https://en.wikipedia.org/wiki/Ampere_(microarchitecture)) |
| RTX 3080 | 8.6 | Desktop-class | 2020 | Ampere| 
| RTX 3070 | 8.6 | Desktop-class | 2020 | Ampere |  
| RTX 3060 Ti | 8.6 | Desktop-class | 2020 | Ampere | 
| H100 | 9.0 | Datacenter-class | 2022 | [Hopper](https://developer.nvidia.com/blog/nvidia-hopper-architecture-in-depth/) | 
| A100 | 8.0 | Datacenter-class | 2020 | Ampere |
| A10 | 8.6 | Datacenter-class | 2021 | Ampere |

GPUs with a compute capability score of 8.0 or above are likely to see the biggest speedups.

And GPUs which are datacenter-class (e.g. A100, A10, H100) are likely to see more significant speedups than desktop-class GPUs (e.g. RTX 3090, RTX 3080, RTX 3070, RTX 3060 Ti).

We can check the compute capability score of our GPU using [`torch.cuda.get_device_capability()`](https://pytorch.org/docs/stable/generated/torch.cuda.get_device_capability.html).

This will output a tuple of `(major, minor)` compute capability scores, for example, `(8, 0)` for the A100.

We'll also get some other details about our GPU such as the name and other info using [`nvidia-smi`](https://developer.nvidia.com/nvidia-system-management-interface).  

> **Resource:** For an in-depth comparison of many different NVIDIA GPUs and their speeds, costs and tradeoffs, I'd recommend reading Tim Dettmers' [*Which GPU for deep learning?*](https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/) blog post.

In [3]:
import torch
import torchvision
# Make sure we're using a NVIDIA GPU
if torch.cuda.is_available():
  gpu_info = !nvidia-smi
  gpu_info = '\n'.join(gpu_info)
  if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU, to leverage the best of PyTorch 2.0, you should connect to a GPU.")

  # Get GPU name
  gpu_name = !nvidia-smi --query-gpu=gpu_name --format=csv
  gpu_name = gpu_name[1]
  GPU_NAME = gpu_name.replace(" ", "_") # remove underscores for easier saving
  print(f'GPU name: {GPU_NAME}')

  # Get GPU capability score
  GPU_SCORE = torch.cuda.get_device_capability()
  print(f"GPU capability score: {GPU_SCORE}")
  if GPU_SCORE >= (8, 0):
    print(f"GPU score higher than or equal to (8, 0), PyTorch 2.x speedup features available.")
  else:
    print(f"GPU score lower than (8, 0), PyTorch 2.x speedup features will be limited (PyTorch 2.x speedups happen most on newer GPUs).")
  
  # Print GPU info
  print(f"GPU information:\n{gpu_info}")

else:
  print("PyTorch couldn't find a GPU, to leverage the best of PyTorch 2.0, you should connect to a GPU.")

GPU name: NVIDIA_GeForce_RTX_3070_Ti
GPU capability score: (8, 6)
GPU score higher than or equal to (8, 0), PyTorch 2.x speedup features available.
GPU information:
Tue Dec 16 16:52:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 Ti     On  |   00000000:01:00.0 Off |                  N/A |
|  0%   31C    P8              4W /  290W |      18MiB /   8192MiB 

### 1.1 Globally set devices

One of my favourite new features in PyTorch 2.x is being able to set the [default device type](https://pytorch.org/tutorials/recipes/recipes/changing_default_device.html ) via:
* Context manager
* Globally

Previously, you could only set the default device type via:
* `tensor.to(device)`

Let's see these two new device settings in action.

In [4]:
import torch

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set the device with context manager (requires PyTorch 2.x+)
with torch.device(device):
    # All tensors created in this block will be on device
    layer = torch.nn.Linear(20, 30)
    print(f"Layer weights are on device: {layer.weight.device}")
    print(f"Layer creating data on device: {layer(torch.randn(128, 20)).device}")

Layer weights are on device: cuda:0
Layer creating data on device: cuda:0


Now how about setting the global device?

This will mean that any tensors created without an explicit device will be created on the device you set by default.

In [5]:
import torch

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Set the device globally
torch.set_default_device(device)

# All tensors created will be on the global device by default
layer = torch.nn.Linear(20, 30)
print(f"Layer weights are on device: {layer.weight.device}")
print(f"Layer creating data on device: {layer(torch.randn(128, 20)).device}")

Layer weights are on device: cuda:0
Layer creating data on device: cuda:0


And now back to CPU.

In [6]:
import torch 

# Set the device globally
torch.set_default_device("cpu")

# All tensors created will be on "cpu"
layer = torch.nn.Linear(20, 30)
print(f"Layer weights are on device: {layer.weight.device}")
print(f"Layer creating data on device: {layer(torch.randn(128, 20)).device}")

Layer weights are on device: cpu
Layer creating data on device: cpu


## 2. Setting up the experiments 

Okay, time to measure speed!

To keep things simple, as we discussed we're going to run a series of four experiments, all with:

* **Model:** ResNet50 (from [TorchVision](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html))
* **Data:** CIFAR10 (from [TorchVision](https://pytorch.org/vision/main/generated/torchvision.datasets.CIFAR10.html))
* **Epochs:** 5 (single run) and 3x5 (multiple runs)
* **Batch size:** 128
* **Image size:** 224

Each experiment will be run with and without `torch.compile()`.

Why the single and multiple runs?

Because we can measure speedups via a single run, however, we'll also want to run the tests multiple times to get an average (just to make sure the results from a single run weren't a fluke or something went wrong). 

> **Note:** Depending on the amount of memory your GPU has, you may have to lower the batch size or the image size. This tutorial is focused on using an NVIDIA A100 GPU with 40GB of memory, the amount of memory on this GPU means it can handle a larger batch size. As of April 2023, NVIDIA A100 GPUs are available via Google Colab Pro. 

Let's start by importing `torch` and `torchvision` and setting the target device.

### 2.1 Create model and transforms

In [ ]:
from pathlib import Path
plant_path = Path("data/plant")
class_names = []
for cls in plant_path.iterdir():
    class_names.append(cls.name)
class_names

['Cassava___brown_streak_disease',
 'Rose___slug_sawfly',
 'Peach___bacterial_spot',
 'Sugercane___red_rot',
 'Bell_pepper___bacterial_spot',
 'Potato___phytophthora',
 'Cassava___green_mottle',
 'Corn___healthy',
 'Potato___nematode',
 'Potato___bacterial_wilt',
 'Lemon___Curl Virus',
 'Lemon___Bacterial Blight',
 'Potato___late_blight',
 'Sugercane___yellow_leaf',
 'Bell_pepper___healthy',
 'Coffee___rust',
 'Apple___gray_spot',
 'Sugercane___rust',
 'Watermelon___downy_mildew',
 'Watermelon___mosaic_virus',
 'Sugercane___healthy',
 'Lemon___Dry Leaf',
 'Grape___black_rot',
 'Grape___Leaf_blight',
 'Potato___healthy',
 'Grape___leaf_blight',
 'Strawberry___leaf_scorch',
 'Cassava___mosaic_disease',
 'Lemon___Healthy Leaf',
 'Coffee___red_spider_mite',
 'Lemon___Deficiency Leaf',
 'Lemon___Sooty Mould',
 'Squash___powdery_mildew',
 'Rice___tungro',
 'Tomato___septoria_leaf_spot',
 'Sugercane___mosaic',
 'Tomato___leaf_mold',
 'Lemon___Anthracnose',
 'Watermelon___healthy',
 'Blueberry

In [ ]:
from timm.data import create_transform
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import timm

rexnet_150_model = timm.create_model("rexnet_150", 
                                     pretrained = True, 
                                     num_classes = len(class_names))

# Create the transform
rexnet_150_transforms = create_transform(input_size=rexnet_150_model.default_cfg["input_size"],
                                         mean=rexnet_150_model.default_cfg.get("mean", IMAGENET_DEFAULT_MEAN),
                                         std=rexnet_150_model.default_cfg.get("std", IMAGENET_DEFAULT_STD),
                                         crop_pct=rexnet_150_model.default_cfg.get("crop_pct", 1.0),
                                         interpolation=rexnet_150_model.default_cfg.get("interpolation", "bilinear")
                                    )

# Load saved weights
rexnet_150_model.load_state_dict(torch.load(f="models/dectection_plant_health_lemon_rexnet150.pth",
                                            map_location=torch.device("cpu"),  # load to CPU
                                           )
                                            )

# Count the number of parameters in the model 
total_params = sum(
    param.numel() for param in rexnet_150_model.parameters() # <- all params
	# param.numel() for param in model.parameters() if param.requires_grad # <- only trainable params
)

print(f"Total parameters of model: {total_params} (the more parameters, the more GPU memory the model will use, the more *relative* of a speedup you'll get)")
print(f"Model transforms:\n{rexnet_150_transforms}")

Total parameters of model: 7963194 (the more parameters, the more GPU memory the model will use, the more *relative* of a speedup you'll get)
Model transforms:
Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


### 2.2 Speedups are most noticeable when a large portion of the GPU is being used

Since modern GPUs are so fast at performing operations, you will often notice the majority of *relative* speedups when as much data as possible is on the GPU.

This can be achieved by:
* **Increasing the batch size** - More samples per batch means more samples on the GPU, for example, using a batch size of 256 instead of 32.
* **Increasing data size** - For example, using larger image size, 224x224 instead of 32x32. A larger data size means that more tensor operations will be happening on the GPU.
* **Increasing model size** - For example, using a larger model such as ResNet101 instead of ResNet50. A larger model means that more tensor operations will be happening on the GPU.
* **Decreasing data transfer** - For example, setting up all your tensors to be on GPU memory, this minimizes the amount of data transfer between the CPU and GPU.

All of these result in *more* data being on the GPU.

<img src="https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/extras-speedups-are-biggest-when-more-gpu-is-used.png" width=950 alt="speedups are biggest when more of the GPU is used"/>

You may be thinking, "but doesn't this mean that the GPU will be slower because it has to do more work?"

This is correct, operations may take longer when using *more* data on the GPU, however, they benefit from [parallelism](https://en.wikipedia.org/wiki/Parallel_computing) (many operations happening at once).

This means that although *more* operations are happening, the GPU is performing as many of them as possible simultaneously.

So while you may see speedups with smaller datasets, models, batch sizes and data sizes, however, you will tend to see the *biggest relative* speedups with increasing scale.



### 2.3 Checking the memory limits of our GPU

To take advantage of speedups at scale, let's check how much memory our GPU has.

If your GPU has less memory, you may need to decrease the batch size or image size (less potential for speedups).

We can check the memory available on our GPU using [`torch.cuda.mem_get_info()`](https://pytorch.org/docs/stable/generated/torch.cuda.mem_get_info.html#torch.cuda.mem_get_info).

This will return a tuple of `(total_free_gpu_memory, total_gpu_memory)`.

Where:
* `total_free_gpu_memory` is the amount of memory currently *not being used* on the GPU in bytes.
* `total_gpu_memory` is the total amount of memory available on the GPU in bytes. 


In [12]:
# Check available GPU memory and total GPU memory 
total_free_gpu_memory, total_gpu_memory = torch.cuda.mem_get_info()
print(f"Total free GPU memory: {round(total_free_gpu_memory * 1e-9, 3)} GB")
print(f"Total GPU memory: {round(total_gpu_memory * 1e-9, 3)} GB")

Total free GPU memory: 7.994 GB
Total GPU memory: 8.221 GB


Wonderful!

The takeaways here are:
1. The higher the memory available on your GPU, **the bigger your batch size can be, the bigger your model can be, the bigger your data samples can be**. 
2. For speedups, you should always be trying to use **as much of the GPU(s) as possible**.

Let's write some code to use a larger batch size if more GPU memory is available.

> **Note:** The ideal batch size you use will depend on the specific GPU and dataset and model you're working with. The code below is specifically targeted for the A100 GPU available on Google Colab Pro. However, you may adjust it for your own GPU. As if you set the batch size too high, you may run into CUDA out of memory errors.

If the total memory on the GPU available is **above 16GB**, let's use a batch size of 128 and an image size of 224 (both of these values can be increased on GPUs with more memory).

If the total memory on the GPU available is **below 16GB**, let's use a batch size of 32 and an image size of 64 (both of these values can be altered on GPUs with less memory).

In [13]:
# Set batch size depending on amount of GPU memory
total_free_gpu_memory_gb = round(total_free_gpu_memory * 1e-9, 3)
if total_free_gpu_memory_gb >= 16:
  BATCH_SIZE = 128 # Note: you could experiment with higher values here if you like.
  IMAGE_SIZE = 224
  print(f"GPU memory available is {total_free_gpu_memory_gb} GB, using batch size of {BATCH_SIZE} and image size {IMAGE_SIZE}")
else:
  BATCH_SIZE = 32
  IMAGE_SIZE = 128
  print(f"GPU memory available is {total_free_gpu_memory_gb} GB, using batch size of {BATCH_SIZE} and image size {IMAGE_SIZE}")

GPU memory available is 7.994 GB, using batch size of 32 and image size 128


Now let's adjust the `transforms` to use the respective `IMAGE_SIZE`.

In [15]:
rexnet_150_transforms = create_transform(
    input_size=(3, IMAGE_SIZE, IMAGE_SIZE),  
    mean=rexnet_150_model.default_cfg.get("mean", IMAGENET_DEFAULT_MEAN),
    std=rexnet_150_model.default_cfg.get("std", IMAGENET_DEFAULT_STD),
    crop_pct=rexnet_150_model.default_cfg.get("crop_pct", 1.0),
    interpolation=rexnet_150_model.default_cfg.get("interpolation", "bilinear")
)
print(f"Updated data transforms:\n{rexnet_150_transforms}")

Updated data transforms:
Compose(
    Resize(size=146, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(128, 128))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


### 2.4  More potential speedups with TF32

TF32 stands for TensorFloat-32, a data format which is a combination of 16-bit and 32-bit floating point numbers.

You can read more about how it works on [NVIDIA's blog](https://blogs.nvidia.com/blog/2020/05/14/tensorfloat-32-precision-format/).

The main thing you should know is that it allows you to **perform faster matrix multiplications** on GPUs with the Ampere architecture and above (a compute capability score of 8.0+).

Although it's not specific to PyTorch 2.0, since we're talking about newer GPUs, it's worth mentioning.

If you're using a GPU with a compute capability score of 8.0 or above, you can enable TF32 by setting [`torch.backends.cuda.matmul.allow_tf32 = True`](https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices) (this defaults to `False`).

Let's write a check that sets it automatically for us based on our GPUs compute capability score.

> **Note:** TensorFloat32 is disabled by default (set to `False`) in PyTorch versions 1.12 onwards. This is because it [may cause inconsistent results across different devices](https://dev-discuss.pytorch.org/t/pytorch-and-tensorfloat32/504). Although this issue is not noticed for all use cases, it's worth knowing. 

In [16]:
if GPU_SCORE >= (8, 0):
  print(f"[INFO] Using GPU with score: {GPU_SCORE}, enabling TensorFloat32 (TF32) computing (faster on new GPUs)")
  torch.backends.cuda.matmul.allow_tf32 = True
else:
  print(f"[INFO] Using GPU with score: {GPU_SCORE}, TensorFloat32 (TF32) not available, to use it you need a GPU with score >= (8, 0)")
  torch.backends.cuda.matmul.allow_tf32 = False

[INFO] Using GPU with score: (8, 6), enabling TensorFloat32 (TF32) computing (faster on new GPUs)


### 2.5 Preparing datasets

In [21]:
from going_modular.going_modular.data_setup import create_dataloaders

train_dataloader, test_dataloader, class_names = create_dataloaders(
    batch_size=BATCH_SIZE,
    transform=rexnet_150_transforms,
    train_dir="data/plant",
)

[INFO] Splitting dataset of length 117501 into splits of size: 94000 (80%), 23501 (19%)
Train data:
Test data:
Classes are not balanced, creating weights for each sample...


In [24]:
import os
NUM_WORKERS = os.cpu_count() # <- use all available CPU cores (this number can be tweaked through experimentation but generally more workers means faster dataloading from CPU to GPU)

# Get the lengths of the datasets
train_len = len(train_dataloader.dataset)
test_len = len(test_dataloader.dataset)

print(f"[INFO] Train dataset length: {train_len}")
print(f"[INFO] Test dataset length: {test_len}")
print(f"Train dataloader length: {len(train_dataloader)} batches of size {BATCH_SIZE}")
print(f"Test dataloader length: {len(test_dataloader)} batches of size {BATCH_SIZE}")
print(f"Using number of workers: {NUM_WORKERS} (generally more workers means faster dataloading from CPU to GPU)")

[INFO] Train dataset length: 94000
[INFO] Test dataset length: 23501
Train dataloader length: 2938 batches of size 32
Test dataloader length: 735 batches of size 32
Using number of workers: 20 (generally more workers means faster dataloading from CPU to GPU)


## 3. Time models across single run

Training and testing functions ready!

Time to start training/evaluating and timing our model.

We'll start with the first experiment. 

### 3.1 Experiment 1 - Single run, no compile

We'll set the number of epochs to `5` and use a learning rate of `0.003` throughout (you can experiment with different learning rates for better results but we're focused on speed).

In [27]:
# Set the number of epochs as a constant
NUM_EPOCHS = 1

# Set the learning rate as a constant (this can be changed to get better results but for now we're just focused on time)
LEARNING_RATE = 0.003

In [28]:
from going_modular.going_modular.engine import train_test_step
rexnet_150_model.to(device)

# Set the loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rexnet_150_model.parameters(), lr=LEARNING_RATE)

single_run_no_compile_results = train_test_step(model=rexnet_150_model,
                                                train_dataloader=train_dataloader,
                                                test_dataloader=test_dataloader,
                                                loss_fn=loss_fn,
                                                optimizer=optimizer,
                                                device=device,
                                                epochs=NUM_EPOCHS
                                                )

  0%|          | 0/1 [00:00<?, ?it/s]

Epoch 0
-------
Looked at 0/94000 samples
Looked at 12800/94000 samples
Looked at 25600/94000 samples
Looked at 38400/94000 samples
Looked at 51200/94000 samples
Looked at 64000/94000 samples
Looked at 76800/94000 samples
Looked at 89600/94000 samples


100%|██████████| 1/1 [05:46<00:00, 346.86s/it]

Train loss: 0.304 | Train accuracy: 90.28% | Max train probability: 92.86%
Test loss: 0.386 | Test accuracy: 86.87% | Max test probability: 90.56%
Train time on cuda: 5.0 minutes 46.9 secondes


### 3.2 Experiment 2 - Single run, with compile

Now we'll do the same experiment but this time we'll use `torch.compile()`.

In [32]:
import importlib
import going_modular.going_modular.utils as utils
importlib.reload(utils)

<module 'going_modular.going_modular.utils' from '/home/jojo/Bureau/Portfolio/Deep Learning/PyTorch/Classification/Computer vision/going_modular/going_modular/utils.py'>

In [33]:
import time
from going_modular.going_modular.utils import create_timm_model
rexnet_150_model, rexnet_150_transforms = create_timm_model(model_name="rexnet_150",
                                                            class_names=class_names,
                                                            IMAGE_SIZE=IMAGE_SIZE)

# Load saved weights
rexnet_150_model.load_state_dict(torch.load(f="models/dectection_plant_health_lemon_rexnet150.pth",
                                            map_location=torch.device("cpu"),  # load to CPU
                                           )
                                            )

rexnet_150_model.to(device)

# Compile the model and time how long it takes
compile_start_time = time.time()

### New in PyTorch 2.x ###
compiled_model = torch.compile(rexnet_150_model)
##########################

compile_end_time = time.time()
compile_time = compile_end_time - compile_start_time
print(f"Time to compile: {compile_time} | Note: The first time you compile your model, the first few epochs will be slower than subsequent runs.")

single_run_with_compile_results = train_test_step(model=compiled_model,
                                                  train_dataloader=train_dataloader,
                                                  test_dataloader=test_dataloader,
                                                  loss_fn=loss_fn,
                                                  optimizer=optimizer,
                                                  device=device,
                                                  epochs=NUM_EPOCHS
                                                  )


Time to compile: 0.7066221237182617 | Note: The first time you compile your model, the first few epochs will be slower than subsequent runs.


  0%|          | 0/1 [00:00<?, ?it/s]

Epoch 0
-------


W1216 18:20:22.877000 419971 torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


Looked at 0/94000 samples
Looked at 12800/94000 samples
Looked at 25600/94000 samples
Looked at 38400/94000 samples
Looked at 51200/94000 samples
Looked at 64000/94000 samples
Looked at 76800/94000 samples
Looked at 89600/94000 samples


100%|██████████| 1/1 [05:06<00:00, 306.82s/it]

Train loss: 0.498 | Train accuracy: 88.94% | Max train probability: 80.15%
Test loss: 0.518 | Test accuracy: 87.95% | Max test probability: 79.47%
Train time on cuda: 5.0 minutes 6.8 secondes


In [35]:
307/347

0.8847262247838616

In [34]:
import torch

if torch.cuda.is_available():
    device = torch.cuda.current_device()
    name = torch.cuda.get_device_name(device)
    major, minor = torch.cuda.get_device_capability(device)
    sm_count = torch.cuda.get_device_properties(device).multi_processor_count
    
    print(f"GPU: {name}")
    print(f"Compute Capability: {major}.{minor}")
    print(f"Number of SMs: {sm_count}")
else:
    print("Aucun GPU détecté")


GPU: NVIDIA GeForce RTX 3070 Ti
Compute Capability: 8.6
Number of SMs: 48


### 3.3 Compare the results of experiment 1 and 2

Nice!

We've got two trained models:

1. One without `torch.compile()`.
2. One with `torch.compile()`.

Let's compare the results of each experiment.

To do so, we'll first create dataframes of the results of each.

Then we'll plot the results of each experiment on a bar chart.

In [36]:
# Turn experiment results into dataframes
import pandas as pd
single_run_no_compile_results_df = pd.DataFrame(single_run_no_compile_results)
single_run_compile_results_df = pd.DataFrame(single_run_with_compile_results)

Got the results for experiments 1 and 2!

Now let's write a function to take in the results and compare them with a bar chart.

We'll add some metadata to the function so it can display some information about the experiments.

Namely all of the parameters in our experiment setup:
* The dataset name.
* The model name.
* The number of epochs.
* The batch size.
* The image size.

In [ ]:
DATASET_NAME = "plant"
MODEL_NAME = "rexnet_150"

In [40]:
# import matplotlib.pyplot as plt
# import numpy as np

# def plot_mean_epoch_times(non_compiled_results: pd.DataFrame, 
#                           compiled_results: pd.DataFrame, 
#                           multi_runs: bool=False, 
#                           num_runs: int=0, 
#                           save: bool=False, 
#                           save_path: str="",
#                           dataset_name: str=DATASET_NAME,
#                           model_name: str=MODEL_NAME,
#                           num_epochs: int=NUM_EPOCHS,
#                           image_size: int=IMAGE_SIZE,
#                           batch_size: int=BATCH_SIZE) -> plt.figure:
    
#     # Get the mean epoch times from the non-compiled models
#     mean_train_epoch_time = non_compiled_results.train_epoch_time.mean()
#     mean_test_epoch_time = non_compiled_results.test_epoch_time.mean()
#     mean_results = [mean_train_epoch_time, mean_test_epoch_time]

#     # Get the mean epoch times from the compiled models
#     mean_compile_train_epoch_time = compiled_results.train_epoch_time.mean()
#     mean_compile_test_epoch_time = compiled_results.test_epoch_time.mean()
#     mean_compile_results = [mean_compile_train_epoch_time, mean_compile_test_epoch_time]

#     # Calculate the percentage difference between the mean compile and non-compile train epoch times
#     train_epoch_time_diff = mean_compile_train_epoch_time - mean_train_epoch_time
#     train_epoch_time_diff_percent = (train_epoch_time_diff / mean_train_epoch_time) * 100

#     # Calculate the percentage difference between the mean compile and non-compile test epoch times
#     test_epoch_time_diff = mean_compile_test_epoch_time - mean_test_epoch_time
#     test_epoch_time_diff_percent = (test_epoch_time_diff / mean_test_epoch_time) * 100

#     # Print the mean difference percentages
#     print(f"Mean train epoch time difference: {round(train_epoch_time_diff_percent, 3)}% (negative means faster)")
#     print(f"Mean test epoch time difference: {round(test_epoch_time_diff_percent, 3)}% (negative means faster)")

#     # Create a bar plot of the mean train and test epoch time for both compiled and non-compiled models
#     plt.figure(figsize=(10, 7))
#     width = 0.3
#     x_indicies = np.arange(len(mean_results))

#     plt.bar(x=x_indicies, height=mean_results, width=width, label="non_compiled_results")
#     plt.bar(x=x_indicies + width, height=mean_compile_results, width=width, label="compiled_results")
#     plt.xticks(x_indicies + width / 2, ("Train Epoch", "Test Epoch"))
#     plt.ylabel("Mean epoch time (seconds, lower is better)")

#     # Create the title based on the parameters passed to the function
#     if multi_runs:
#         plt.suptitle("Multiple run results")
#         plt.title(f"GPU: {gpu_name} | Epochs: {num_epochs} ({num_runs} runs) | Data: {dataset_name} | Model: {model_name} | Image size: {image_size} | Batch size: {batch_size}")
#     else:
#         plt.suptitle("Single run results")
#         plt.title(f"GPU: {gpu_name} | Epochs: {num_epochs} | Data: {dataset_name} | Model: {model_name} | Image size: {image_size} | Batch size: {batch_size}")
#     plt.legend();

#     # Save the figure
#     if save:
#         assert save_path != "", "Please specify a save path to save the model figure to via the save_path parameter."
#         plt.savefig(save_path)
#         print(f"[INFO] Plot saved to {save_path}")

In [41]:
# # Create directory for saving figures
# import os
# dir_to_save_figures_in = "pytorch_2_results/figures/" 
# os.makedirs(dir_to_save_figures_in, exist_ok=True)

# # Create a save path for the single run results
# save_path_multi_run = f"{dir_to_save_figures_in}single_run_{GPU_NAME}_{MODEL_NAME}_{DATASET_NAME}_{IMAGE_SIZE}_train_epoch_time.png"
# print(f"[INFO] Save path for single run results: {save_path_multi_run}")

# # Plot the results and save the figures
# plot_mean_epoch_times(non_compiled_results=single_run_no_compile_results_df, 
#                       compiled_results=single_run_compile_results_df, 
#                       multi_runs=False, 
#                       save_path=save_path_multi_run, 
#                       save=True)

## 4. Time models across multiple runs

Now we've tested our model with a single run with `torch.compile()` on and off, let's do the same for multiple runs.

We're going to start by creating three functions for experiments 3 and 4.

1. **Experiment 3:** `create_and_train_non_compiled_model()` - this function will be similar to the workflow we've used for the single runs. We'll put the model creation (via `create_model()`) and training in a single function so we can call it multiple times (for multiple runs) and measure the time of each run.
2. **Experiment 4:** `create_compiled_model()` - this function will be similar to the `create_model()` function above, however, it will create a normal PyTorch model and then call `torch.compile()` on it and return it.
3. **Experiment 4:** `train_compiled_model()` - this function will take in a compiled model and train it in the same way we've been training our models for single runs.

Why separate functions 2 and 3 (`create_compiled_model()` and `train_compiled_model()`) for experiment 4?

Because calling `torch.compile()` on model means that for the first few runs, the model will be "warming up" as PyTorch calculates a bunch of optimization steps behind the scenes.

So in practice, you'll generally want to compile up front *once* and then train/perform inference with an already compiled model.